### LLMChains & OutputParsers

Instead of just using models you can combine a model and a prompt. This can be done with the LLMChain class.
We will additional, more complex chains in this notebook

In [15]:
! pip install langchain_google_genai
import sys
!{sys.executable} -m pip install langchain langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!git clone https://github.com/sridba503/LangChain.git
%cd /content/LangChain
!pwd

Cloning into 'LangChain'...
remote: Enumerating objects: 598, done.
remote: Counting objects: 100% (598/598), done.
remote: Compressing objects: 100% (269/269), done.
remote: Total 598 (delta 289), reused 561 (delta 273), pack-reused 0 (from 0)
Receiving objects: 100% (598/598), 7.92 MiB | 25.18 MiB/s, done.
Resolving deltas: 100% (289/289), done.
/content/LangChain
/content/LangChain


In [3]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [5]:
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

# Retrieve the API key from Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Configure the Generative AI library with your API key
genai.configure(api_key=GOOGLE_API_KEY)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [26]:
TEMPLATE = """
Interprete the text and evaluate the text.
sentiment: is the text in a positive, neutral or negative sentiment? Sentiment is required.
subject: What subject is the text about? Use exactly one word. Use 'None' if no subject was provided.
price: How much did the customer pay? Use 'None' if no price was provided.

Format the output as JSON with the following keys:
sentiment
subject
price

text: {input}
"""

In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains.llm import LLMChain

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite",temperature=0.76, google_api_key=GOOGLE_API_KEY)

prompt_template = ChatPromptTemplate.from_template(template=TEMPLATE)
chain = LLMChain(llm=llm, prompt=prompt_template)
chain.invoke(input="I ordered pizza salami from the restaurant Bellavista. It was ok, but the dough could have been a bit more crisp.")

{'input': 'I ordered pizza salami from the restaurant Bellavista. It was ok, but the dough could have been a bit more crisp.',
 'text': '```json\n{\n  "sentiment": "neutral",\n  "subject": "pizza",\n  "price": "None"\n}\n```'}

### Response Schemas

There were two issues with the output: The output also contains text and the output is just a string, not a dictionary.

In [28]:
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser

response_schemas = [
    ResponseSchema(name="sentiment", description="is the text in a positive, neutral or negative sentiment? Sentiment is required."),
    ResponseSchema(name="subject", description="What subject is the text about? Use exactly one word. Use None if no price was provided."),
    ResponseSchema(name="price", description="How much did the customer pay? Use None if no price was provided.", type="float")
]
print(response_schemas)

[ResponseSchema(name='sentiment', description='is the text in a positive, neutral or negative sentiment? Sentiment is required.', type='string'), ResponseSchema(name='subject', description='What subject is the text about? Use exactly one word. Use None if no price was provided.', type='string'), ResponseSchema(name='price', description='How much did the customer pay? Use None if no price was provided.', type='float')]


In [29]:
parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = parser.get_format_instructions()
print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"sentiment": string  // is the text in a positive, neutral or negative sentiment? Sentiment is required.
	"subject": string  // What subject is the text about? Use exactly one word. Use None if no price was provided.
	"price": float  // How much did the customer pay? Use None if no price was provided.
}
```


In [34]:
# Create prompt template
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

# Redefine prompt with correct message structure
prompt = ChatPromptTemplate(
    messages=[
        SystemMessagePromptTemplate.from_template(
            "Interprete the text and evaluate the text. "
            "sentiment: is the text in a positive, neutral or negative sentiment? "
            "subject: What subject is the text about? Use exactly one word. "
            "Just return the JSON, do not add ANYTHING, NO INTERPRETATION! \n"
            "{format_instructions}\n"
        ),
        HumanMessagePromptTemplate.from_template("text: {input}")
    ],
    input_variables=["input"],
    partial_variables={"format_instructions": format_instructions}
)


In [35]:


_input = prompt.format_prompt(input="I ordered pizza salami from the restaurant Bellavista. It was ok, but the dough could have been a bit more crisp.")
output = llm.invoke(_input.to_messages())
print(output.content)

[{'type': 'text', 'text': '```json\n{\n\t"sentiment": "neutral",\n\t"subject": "Pizza",\n\t"price": null\n}\n```', 'extras': {'signature': 'EjQKMgERTTIPyGBJOa7AlTy1NR6rvISaLAH6tz2kgctWBUviutNaFEUsMM5bexKcrgjJ7bhD'}}]


In [37]:
json_output = parser.parse(output.content[0]['text'])
print(json_output)

{'sentiment': 'neutral', 'subject': 'Pizza', 'price': None}


In [38]:
json_output.get("sentiment")

'neutral'